In [1]:
import pandas as pd
import numpy as np
import json
import os

In [2]:
# Carpeta donde estan los datos ya consolidados
carpeta_consolidados = os.path.join("..", "datasets_consolidados")

# Carpeta donde tambien estan algunos datos reales que no pasan por consolidacion
carpeta_datasets = os.path.join("..", "datasets")

# Carpeta para guardar documentos de salida (reportes, diccionario, catalogo)
carpeta_documentacion = os.path.join("..", "documentacion")
os.makedirs(carpeta_documentacion, exist_ok=True)

# Aqui vamos guardando el resultado de cada prueba (PASA / FALLA)
reporte_validacion = []

In [3]:
########################## CREAR CATALOGO DE EVENTOS (paso previo) ##########################

In [4]:
# Los tickets tienen un ID_Evento, pero no existe ningun catalogo real de eventos
# en el proyecto. Creamos uno a partir de los IDs unicos que aparecen en tickets,
# para poder validar despues que todo ID_Evento sea valido (integridad referencial).
tickets_df = pd.read_csv(os.path.join(carpeta_consolidados, "tickets_consolidados.csv"))

ids_evento_unicos = sorted(tickets_df["ID_Evento"].unique())
catalogo_eventos_df = pd.DataFrame({
    "ID_Evento": ids_evento_unicos,
    "Nombre_Evento": [f"Evento_{i}" for i in ids_evento_unicos]
})

catalogo_eventos_df.to_csv(os.path.join(carpeta_documentacion, "catalogo_eventos.csv"), index=False)
catalogo_eventos_df

,ID_Evento,Nombre_Evento
0,100,Evento_100
1,101,Evento_101
2,102,Evento_102
3,103,Evento_103
4,104,Evento_104
...,...,...
395,495,Evento_495
396,496,Evento_496
397,497,Evento_497
398,498,Evento_498


In [5]:
########################## PRUEBA 1: CONTEO DE REGISTROS ##########################

In [6]:
# Verificar que cada archivo consolidado tenga un numero de filas razonable (mayor a 0)
biometria_df = pd.read_json(os.path.join(carpeta_consolidados, "biometria_consolidada.json"))
movilidad_df = pd.read_csv(os.path.join(carpeta_consolidados, "movilidad_consolidada.csv"))
paris_df = pd.read_csv(os.path.join(carpeta_consolidados, "paris_resultados_consolidados.csv"))

conteos = {
    "Tickets": len(tickets_df),
    "Biometria": len(biometria_df),
    "Movilidad": len(movilidad_df),
    "ParisResultados": len(paris_df)
}

for nombre, filas in conteos.items():
    resultado = "PASA" if filas > 0 else "FALLA"
    reporte_validacion.append({"Prueba": "Conteo de registros", "Dataset": nombre, "Detalle": f"{filas} filas", "Resultado": resultado})

print(conteos)

{'Tickets': 1000000, 'Biometria': 1000000, 'Movilidad': 1000000, 'ParisResultados': 21316}


In [7]:
########################## PRUEBA 2: RANGOS VALIDOS ##########################

In [8]:
# BPM debe estar entre 40 y 220 (rango fisiologico humano posible)
bpm_fuera_de_rango = ((biometria_df["BPM"] < 40) | (biometria_df["BPM"] > 220)).sum()
resultado = "PASA" if bpm_fuera_de_rango == 0 else "FALLA"
reporte_validacion.append({"Prueba": "Rango valido", "Dataset": "Biometria (BPM 40-220)", "Detalle": f"{bpm_fuera_de_rango} filas fuera de rango", "Resultado": resultado})
print("BPM fuera de rango:", bpm_fuera_de_rango)

BPM fuera de rango: 0


In [9]:
# Precio_USD no puede ser negativo
precios_negativos = (tickets_df["Precio_USD"] < 0).sum()
resultado = "PASA" if precios_negativos == 0 else "FALLA"
reporte_validacion.append({"Prueba": "Rango valido", "Dataset": "Tickets (Precio_USD >= 0)", "Detalle": f"{precios_negativos} precios negativos", "Resultado": resultado})
print("Precios negativos:", precios_negativos)

Precios negativos: 0


In [10]:
########################## PRUEBA 3: UNICIDAD ##########################

In [11]:
# ID_Ticket no debe repetirse, es la llave primaria del dataset
ids_ticket_repetidos = tickets_df["ID_Ticket"].duplicated().sum()
resultado = "PASA" if ids_ticket_repetidos == 0 else "FALLA"
reporte_validacion.append({"Prueba": "Unicidad", "Dataset": "Tickets (ID_Ticket)", "Detalle": f"{ids_ticket_repetidos} IDs repetidos", "Resultado": resultado})
print("ID_Ticket repetidos:", ids_ticket_repetidos)

ID_Ticket repetidos: 0


In [12]:
# ID_Lectura no debe repetirse, es la llave primaria de biometria
ids_lectura_repetidos = biometria_df["ID_Lectura"].duplicated().sum()
resultado = "PASA" if ids_lectura_repetidos == 0 else "FALLA"
reporte_validacion.append({"Prueba": "Unicidad", "Dataset": "Biometria (ID_Lectura)", "Detalle": f"{ids_lectura_repetidos} IDs repetidos", "Resultado": resultado})
print("ID_Lectura repetidos:", ids_lectura_repetidos)

ID_Lectura repetidos: 0


In [13]:
########################## PRUEBA 4: INTEGRIDAD REFERENCIAL ##########################

In [14]:
# Todo ID_Evento de tickets debe existir en el catalogo de eventos
eventos_invalidos = (~tickets_df["ID_Evento"].isin(catalogo_eventos_df["ID_Evento"])).sum()
resultado = "PASA" if eventos_invalidos == 0 else "FALLA"
reporte_validacion.append({"Prueba": "Integridad referencial", "Dataset": "Tickets -> Catalogo_Eventos", "Detalle": f"{eventos_invalidos} IDs de evento invalidos", "Resultado": resultado})
print("IDs de evento invalidos:", eventos_invalidos)

IDs de evento invalidos: 0


In [15]:
# Todo pais que aparece en resultados de Paris 2024 debe existir en el catalogo de NOCs
nocs_df = pd.read_csv(os.path.join(carpeta_datasets, "paris 2024", "nocs.csv"))

paises_invalidos = (~paris_df["participant_country_code"].isin(nocs_df["code"])).sum()
resultado = "PASA" if paises_invalidos == 0 else "FALLA"
reporte_validacion.append({"Prueba": "Integridad referencial", "Dataset": "ParisResultados -> Nocs", "Detalle": f"{paises_invalidos} codigos de pais invalidos", "Resultado": resultado})
print("Codigos de pais invalidos:", paises_invalidos)

Codigos de pais invalidos: 0


In [16]:
########################## PRUEBA 5: CONSISTENCIA DE FECHAS ##########################

In [17]:
# Fecha_Compra de tickets debe estar dentro del periodo en que se vendieron entradas
# (segun el generador, entre 2024-07-01 y 2024-08-15)
tickets_df["Fecha_Compra"] = pd.to_datetime(tickets_df["Fecha_Compra"])

fecha_min = pd.Timestamp("2024-07-01")
fecha_max = pd.Timestamp("2024-08-15")

fechas_invalidas = ((tickets_df["Fecha_Compra"] < fecha_min) | (tickets_df["Fecha_Compra"] > fecha_max)).sum()
resultado = "PASA" if fechas_invalidas == 0 else "FALLA"
reporte_validacion.append({"Prueba": "Consistencia de fechas", "Dataset": "Tickets (Fecha_Compra)", "Detalle": f"{fechas_invalidas} fechas fuera de rango", "Resultado": resultado})
print("Fechas invalidas en tickets:", fechas_invalidas)

Fechas invalidas en tickets: 0


In [18]:
# La fecha de cada resultado de Paris 2024 debe caer dentro de las fechas oficiales del evento
paris_df["date"] = pd.to_datetime(paris_df["date"], errors="coerce", utc=True)

fecha_min_juegos = pd.Timestamp("2024-07-24", tz="UTC")
fecha_max_juegos = pd.Timestamp("2024-08-11", tz="UTC")

fechas_fuera_de_juegos = ((paris_df["date"] < fecha_min_juegos) | (paris_df["date"] > fecha_max_juegos)).sum()
resultado = "PASA" if fechas_fuera_de_juegos == 0 else "FALLA"
reporte_validacion.append({"Prueba": "Consistencia de fechas", "Dataset": "ParisResultados (date)", "Detalle": f"{fechas_fuera_de_juegos} fechas fuera del periodo oficial", "Resultado": resultado})
print("Fechas fuera del periodo de los Juegos:", fechas_fuera_de_juegos)

Fechas fuera del periodo de los Juegos: 435


In [19]:
########################## REPORTE FINAL DE VALIDACION ##########################

In [20]:
reporte_df = pd.DataFrame(reporte_validacion)
reporte_df.to_csv(os.path.join(carpeta_documentacion, "reporte_validacion.csv"), index=False)
reporte_df

,Prueba,Dataset,Detalle,Resultado
0,Conteo de registros,Tickets,1000000 filas,PASA
1,Conteo de registros,Biometria,1000000 filas,PASA
2,Conteo de registros,Movilidad,1000000 filas,PASA
3,Conteo de registros,ParisResultados,21316 filas,PASA
4,Rango valido,Biometria (BPM 40-220),0 filas fuera de rango,PASA
5,Rango valido,Tickets (Precio_USD >= 0),0 precios negativos,PASA
6,Unicidad,Tickets (ID_Ticket),0 IDs repetidos,PASA
7,Unicidad,Biometria (ID_Lectura),0 IDs repetidos,PASA
8,Integridad referencial,Tickets -> Catalogo_Eventos,0 IDs de evento invalidos,PASA
9,Integridad referencial,ParisResultados -> Nocs,0 codigos de pais invalidos,PASA
